In [2]:
import requests
import pandas as pd

# Test single point first to check if API is reachable
print("Testing connection to ISRIC...")

url = "https://rest.isric.org/soilgrids/v2.0/properties/query?lon=35.2699&lat=0.5204&property=phh2o&depth=0-30cm&value=mean"

try:
    r = requests.get(url, timeout=15)
    print(f"Status code: {r.status_code}")
    print(r.text[:300])
except Exception as e:
    print(f"Connection failed: {e}")

Testing connection to ISRIC...
Status code: 200
{"type":"Feature","geometry":{"type":"Point","coordinates":[35.2699,0.5204]},"properties":{"layers":[]},"query_time_s":0.00922393798828125}


In [3]:
import requests
import pandas as pd

print("Fetching soil data via ISRIC WCS endpoint...")

# Alternative: use SoilGrids REST with wider coverage
points = {
    "Eldoret":     (0.5204, 35.2699),
    "Turbo":       (0.6297, 35.0416),
    "Soy":         (0.5700, 35.1200),
    "BurntForest": (0.2700, 35.3400),
    "Moiben":      (0.7100, 35.1100),
}

# Try all depths and properties separately
results = []
for name, (lat, lon) in points.items():
    print(f"  → {name}...")
    row = {"Location": name}
    
    for prop in ["phh2o", "nitrogen", "clay", "soc"]:
        for depth in ["0-5cm", "5-15cm", "15-30cm", "0-30cm"]:
            url = (
                f"https://rest.isric.org/soilgrids/v2.0/properties/query"
                f"?lon={lon}&lat={lat}"
                f"&property={prop}"
                f"&depth={depth}"
                f"&value=mean"
            )
            try:
                r = requests.get(url, timeout=20).json()
                layers = r.get("properties", {}).get("layers", [])
                if layers:
                    val = layers[0]["depths"][0]["values"]["mean"]
                    if val is not None:
                        row[f"{prop}_{depth}"] = val
                        print(f"    ✓ {prop} at {depth}: {val}")
                        break  # use first depth that returns data
            except:
                pass
    
    results.append(row)

df = pd.DataFrame(results)
print("\nResults:")
print(df)

Fetching soil data via ISRIC WCS endpoint...
  → Eldoret...
  → Turbo...
    ✓ phh2o at 0-5cm: 61
    ✓ nitrogen at 0-5cm: 360
    ✓ clay at 0-5cm: 392
    ✓ soc at 0-5cm: 374
  → Soy...
    ✓ phh2o at 0-5cm: 60
    ✓ nitrogen at 0-5cm: 310
    ✓ clay at 0-5cm: 394
    ✓ soc at 0-5cm: 328
  → BurntForest...
    ✓ phh2o at 0-5cm: 60
    ✓ nitrogen at 0-5cm: 320
    ✓ clay at 0-5cm: 427
    ✓ soc at 0-5cm: 426
  → Moiben...
    ✓ phh2o at 0-5cm: 61
    ✓ nitrogen at 0-5cm: 308
    ✓ clay at 0-5cm: 387
    ✓ soc at 0-5cm: 346

Results:
      Location  phh2o_0-5cm  nitrogen_0-5cm  clay_0-5cm  soc_0-5cm
0      Eldoret          NaN             NaN         NaN        NaN
1        Turbo         61.0           360.0       392.0      374.0
2          Soy         60.0           310.0       394.0      328.0
3  BurntForest         60.0           320.0       427.0      426.0
4       Moiben         61.0           308.0       387.0      346.0


In [4]:
# ISRIC returns scaled integers — convert to real units
# phh2o: divide by 10 → pH (e.g. 61 → 6.1)
# nitrogen: divide by 100 → g/kg (e.g. 360 → 3.60)
# clay: divide by 10 → % (e.g. 392 → 39.2%)
# soc: divide by 10 → g/kg (e.g. 374 → 37.4)

df_soil = df.dropna().copy()  # drop Eldoret (no data)

df_soil["Soil_pH"]          = df_soil["phh2o_0-5cm"]    / 10
df_soil["Nitrogen_gkg"]     = df_soil["nitrogen_0-5cm"] / 100
df_soil["Clay_Pct"]         = df_soil["clay_0-5cm"]     / 10
df_soil["Organic_Carbon"]   = df_soil["soc_0-5cm"]      / 10

# Drop raw scaled columns
df_soil = df_soil[["Location","Soil_pH","Nitrogen_gkg","Clay_Pct","Organic_Carbon"]]

print("Converted soil values:")
print(df_soil.to_string(index=False))

# County average — one row for master dataset
avg = df_soil.drop(columns="Location").mean()
df_soil_avg = pd.DataFrame([avg])
df_soil_avg.insert(0, "Location", "Uasin_Gishu_Average")

print("\n✓ County average:")
print(df_soil_avg.to_string(index=False))

# Save
df_soil.to_csv("../data/raw/soil_raw.csv", index=False)
df_soil_avg.to_csv("../data/processed/soil_clean.csv", index=False)
print("\n✓ Saved soil_raw.csv and soil_clean.csv")

Converted soil values:
   Location  Soil_pH  Nitrogen_gkg  Clay_Pct  Organic_Carbon
      Turbo      6.1          3.60      39.2            37.4
        Soy      6.0          3.10      39.4            32.8
BurntForest      6.0          3.20      42.7            42.6
     Moiben      6.1          3.08      38.7            34.6

✓ County average:
           Location  Soil_pH  Nitrogen_gkg  Clay_Pct  Organic_Carbon
Uasin_Gishu_Average     6.05         3.245      40.0           36.85

✓ Saved soil_raw.csv and soil_clean.csv


 Soil interpretation:

pH 6.05 — slightly acidic, actually ideal for maize (optimal is 5.8–6.5) 
Nitrogen 3.25 g/kg — good natural fertility for highland Kenya 
Clay 40% — heavy clay soil, typical Nitisol, retains water well 
Organic Carbon 36.85 g/kg — very high, excellent for soil health 

This confirms Uasin Gishu has some of Kenya's best maize-growing soils
